In [5]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [6]:
import os

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [7]:
def llm(prompt):
    response = openai_client.responses.create(
        model="openai/gpt-oss-120b",
        input=prompt
    )
    return response.output_text

In [9]:
llm("Hey, what's up?")

"Hey! I'm doing great—thanks for asking. How can I help you today?"

In [5]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Yes, you can enroll now. Which course are you interested in, and do you have any prerequisites or scheduling preferences?


In [6]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [7]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [8]:
question = "I just discovered the course. Can I join now?"
answer = llm(prompt)
print(answer)

Yes—you can still join the course.  Just keep in mind that if you want to earn a certificate you’ll need to submit your project before the submission deadline (i.e., while the course is still accepting projects).


In [9]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [11]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1342

In [12]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [13]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [25]:
search_results = index.search(
  question, 
  boost_dict={"question": 2.0, "section": 0.5},
  filter_dict={"course": "llm-zoomcamp"},  
  num_results=5,
  )

In [34]:
def search(question, course="llm-zoomcamp"):
   boost_dict = {"question": 2.0, "section": 0.5}
   filter_dict = {"course": course}

   return index.search(
     question, 
     boost_dict=boost_dict,
     filter_dict=filter_dict,
     num_results=5
  )

In [37]:
search_results = search(question)
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [38]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [44]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [41]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [43]:
context = build_context(search_results)
print(context)

General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.

You can only peer-review projects at the time the course is run

In [47]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [48]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [49]:
response = openai_client.responses.create(
        model="openai/gpt-oss-120b",
        input=prompt
    )

In [50]:
response.output_text

'**Answer:** Yes—you can still join the course.  \n\nIf you want to earn a certificate, make sure you submit your capstone project **before the submission deadline** (i.e., while we’re still accepting project submissions). After that point you can continue learning, but a certificate will no longer be available.'

In [51]:
print(response.model_dump_json(indent=2))

{
  "id": "resp_01ktsamcjkea3aqm6105zn83fp",
  "created_at": 1781113958.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "openai/gpt-oss-120b",
  "object": "response",
  "output": [
    {
      "id": "resp_01ktsamcjkea3rxe588th5jcxe",
      "summary": [],
      "type": "reasoning",
      "content": [
        {
          "text": "The user asks: \"Question: I just discovered the course. Can I join now? Context: ...\" So they want answer. According to context, answer: Yes, you can join, but need to submit project while submissions still accepted to get certificate. Provide that.",
          "type": "reasoning_text"
        }
      ],
      "encrypted_content": null,
      "status": "completed"
    },
    {
      "id": "msg_01ktsamcjkea4a9kbdbqkzhb4h",
      "content": [
        {
          "annotations": [],
          "text": "**Answer:** Yes—you can still join the course.  \n\nIf you want to earn a certificate, make sure you submit yo

In [52]:
response.usage

ResponseUsage(input_tokens=399, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=127, output_tokens_details=OutputTokensDetails(reasoning_tokens=55), total_tokens=526)

In [53]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.00087075

In [55]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = openai_client.responses.create(
        model="openai/gpt-oss-120b",
        input=message_history
    )

In [56]:
response.output_text

'Yes—you can still join the course. If you’d like to earn a certificate, just make sure to submit your capstone project before we stop accepting submissions. (You can start learning and doing the homework right away.)'

In [57]:
def llm(instructions, user_prompt, model="openai/gpt-oss-120b"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
            model=model,
            input=message_history
        )

    return response.output_text

In [59]:
def rag(query, model="openai/gpt-oss-120b"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [60]:
answer = rag(question)
print(answer)

Yes—you can still join the course. If you want to earn a certificate, just make sure you submit your project while we’re still accepting submissions.
